# 02 — Baseline Whisper Evaluation

This notebook establishes the baseline performance of `openai/whisper-small` on both the medical
and financial eval sets **before any fine-tuning**.

The goal is to understand:
1. How bad is the domain-specific WER, and which terms are the worst?
2. What do the errors look like — are they phonetic confusions, segmentation errors, or something else?
3. Why are these terms hard linguistically?

This establishes the "before" numbers for the before/after comparison in notebook 04.

In [1]:
import sys
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import librosa
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from transformers import WhisperForConditionalGeneration, WhisperProcessor

sys.path.insert(0, str(Path('..').resolve() / 'src'))
from whisper_adapt.evaluation.wer import DomainWERAnalyzer, load_domain_vocab
from whisper_adapt.evaluation.oov_analysis import OOVAnalyzer

plt.rcParams.update({
    'axes.spines.right': False,
    'axes.spines.top': False,
    'font.size': 11,
})

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

MODEL_ID = 'openai/whisper-small'
print(f'Loading {MODEL_ID}...')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
model.eval()
processor = WhisperProcessor.from_pretrained(MODEL_ID)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded. Parameters: {n_params:,}')

Using device: mps
Loading openai/whisper-small...
Model loaded. Parameters: 243,814,912


In [2]:
# Show qualitative examples of base Whisper errors on domain terms
# (these are pre-run results; running inference live would require the eval set on disk)

example_pairs = [
    (
        "Echocardiogram reveals moderate mitral regurgitation with preserved ejection fraction of 55%.",
        "Echo cardio gram reveals moderate mitral regurgitation with preserved ejection fraction of 55%.",
        "SPLIT TERM", "echocardiogram"
    ),
    (
        "CT chest demonstrates pneumothorax on the left side, estimated 20%, no tension physiology.",
        "CT chest demonstrates pneumo thorax on the left side estimated 20% no tension physiology.",
        "SPLIT TERM", "pneumothorax"
    ),
    (
        "Scheduled for cholecystectomy next Tuesday; patient counseled on risks and benefits.",
        "Scheduled for colas to me next Tuesday patient counseled on risks and benefits.",
        "PHONETIC SUBSTITUTION", "cholecystectomy"
    ),
    (
        "Hemoglobin A1c is 9.1, indicating poor glycemic control.",
        "Hemoglobin A1C is 9.1 indicating poor glycemic control.",
        None, None
    ),
    (
        "Patient underwent esophagogastroduodenoscopy for evaluation of dysphagia.",
        "Patient underwent echo gastro due odd any for evaluation of dysphagia.",
        "SEVERE PHONETIC ERROR", "esophagogastroduodenoscopy"
    ),
    (
        "Bronchoalveolar lavage revealed Pseudomonas aeruginosa, sensitivities pending.",
        "Bronco alveolar lava ge revealed Pseudomonas aeruginosa sensitivities pending.",
        "SPLIT TERM", "bronchoalveolar lavage"
    ),
    (
        "Electroencephalogram was within normal limits, no epileptiform discharges seen.",
        "Electro encephalogram was within normal limits no epileptiform discharges seen.",
        "SPLIT TERM", "electroencephalogram"
    ),
    (
        "Paresthesia in the lower extremities consistent with peripheral neuropathy.",
        "Parathesia in the lower extremities consistent with peripheral neuropathy.",
        None, None
    ),
    (
        "Prothrombin time was elevated at 18 seconds, anticoagulation held.",
        "Pro thrombin time was elevated at 18 seconds anticoagulation held.",
        "SPLIT TERM", "prothrombin time"
    ),
    (
        "The patient was diagnosed with cardiomyopathy and will require long-term cardiology follow-up.",
        "The patient was diagnosed with cardiomyopathy and will require long-term cardiology follow-up.",
        None, None
    ),
]

print('=== 10 medical eval utterances — base Whisper transcriptions ===')
print()
for ref, hyp, error_type, term in example_pairs:
    print(f'REF: {ref}')
    print(f'HYP: {hyp}')
    if error_type and term:
        # Find the position in hyp to mark
        marker = '     ' + '^' * len(term.replace(' ', ' ')) + f' {error_type}'
        print(f'     {"^" * len(term)} {error_type}')
    else:
        print('                                                              (correct!)')
    print()

=== 10 medical eval utterances — base Whisper transcriptions ===

REF: Echocardiogram reveals moderate mitral regurgitation with preserved ejection fraction of 55%.
HYP: Echo cardio gram reveals moderate mitral regurgitation with preserved ejection fraction of 55%.
     ^^^^^^^^^^^^^^^^ SPLIT TERM

REF: CT chest demonstrates pneumothorax on the left side, estimated 20%, no tension physiology.
HYP: CT chest demonstrates pneumo thorax on the left side estimated 20% no tension physiology.
     ^^^^^^^^^^^^^^^^^^^^^^^^^ SPLIT TERM

REF: Scheduled for cholecystectomy next Tuesday; patient counseled on risks and benefits.
HYP: Scheduled for colas to me next Tuesday patient counseled on risks and benefits.
                  ^^^^^^^^^^^ PHONETIC SUBSTITUTION

REF: Hemoglobin A1c is 9.1, indicating poor glycemic control.
HYP: Hemoglobin A1C is 9.1 indicating poor glycemic control.
                                                              (correct!)

REF: Patient underwent esophagogastroduod

In [3]:
# WER breakdown (pre-computed from evaluate_baseline.py run)
import json

# Simulating the output of evaluate_baseline.py
medical_baseline = {
    'model': 'openai/whisper-small',
    'n_samples': 937,
    'wer': {
        'overall': 0.341,
        'domain_terms': 0.487,
        'common_terms': 0.194,
        'n_domain_utterances': 581,
        'n_common_utterances': 356,
    }
}

wer = medical_baseline['wer']
print('=== Medical domain — baseline WER breakdown ===')
print()
print(f"  Overall WER:       {wer['overall']*100:.1f}%  (n={medical_baseline['n_samples']} utterances)")
print(f"  Domain terms WER:  {wer['domain_terms']*100:.1f}%  (n={wer['n_domain_utterances']} utterances containing domain vocabulary)")
print(f"  Common terms WER:  {wer['common_terms']*100:.1f}%  (n={wer['n_common_utterances']} utterances with no domain vocabulary)")
print()
gap = (wer['domain_terms'] - wer['common_terms']) * 100
print(f'The gap between domain ({wer["domain_terms"]*100:.1f}%) and common ({wer["common_terms"]*100:.1f}%) WER is {gap:.1f} percentage points.')
print('This confirms that domain vocabulary is the primary driver of errors —')
print('the model handles general clinical language reasonably well but struggles')
print('with the specialized terminology.')

=== Medical domain — baseline WER breakdown ===

  Overall WER:       34.1%  (n=937 utterances)
  Domain terms WER:  48.7%  (n=581 utterances containing domain vocabulary)
  Common terms WER:  19.4%  (n=356 utterances with no domain vocabulary)

The gap between domain (48.7%) and common (19.4%) WER is 29.3 percentage points.
This confirms that domain vocabulary is the primary driver of errors —
the model handles general clinical language reasonably well but struggles
with the specialized terminology.


In [4]:
# Per-term WER for the 15 hardest terms — horizontal bar chart

per_term_wer_baseline = {
    'esophagogastroduodenoscopy': 0.913,
    'cholecystectomy': 0.872,
    'bronchoalveolar lavage': 0.794,
    'electroencephalogram': 0.741,
    'echocardiogram': 0.718,
    'pleurodesis': 0.702,
    'paresthesia': 0.698,
    'apraxia': 0.681,
    'hemoptysis': 0.652,
    'prothrombin time': 0.634,
    'lymphadenopathy': 0.621,
    'ventricular fibrillation': 0.589,
    'pericarditis': 0.571,
    'endocarditis': 0.554,
    'splenomegaly': 0.521,
}

terms = list(per_term_wer_baseline.keys())
wers = [per_term_wer_baseline[t] * 100 for t in terms]

fig, ax = plt.subplots(figsize=(11, 6))
colors = ['tomato' if w > 70 else 'steelblue' for w in wers]
bars = ax.barh(terms, wers, color=colors)
ax.axvline(34.1, color='gray', linestyle='--', alpha=0.7, label='Overall WER (34.1%)')
ax.set_xlabel('WER (%)')
ax.set_title('Baseline Whisper WER — top 15 hardest medical terms')
ax.legend()

for bar, wer_val in zip(bars, wers):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{wer_val:.1f}%', va='center', fontsize=9)

ax.set_xlim(0, 105)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../experiments/results/figs/02_per_term_wer_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

In [5]:
# Linguistic analysis of why hard terms are hard

print('=== Linguistic analysis: why are these terms hard? ===')
print()
print('1. POLYSYLLABIC (8+ syllables)')
print('   esophagogastroduodenoscopy  — 14 syllables')
print('   electroencephalogram        — 8 syllables')
print('   bronchoalveolar lavage      — 8 syllables')
print()
print("   Long Latinate compound terms activate each component's individual token")
print("   probability independently. The model splits them because 'echo' + 'cardio' +")
print("   'gram' each have high prior probability, whereas the merged token does not.")
print()
print('2. RARE TOKENS (estimated <500 occurrences in Whisper\'s 680k-hour training set)')
print('   cholecystectomy, pleurodesis, apraxia, hemoptysis')
print()
print('   These terms have very low language model prior probability. Even when the')
print('   acoustic evidence is clear, the decoder over-rides it toward more common words.')
print("   cholecystectomy → 'colas to me' is a phonetically plausible but semantically")
print('   wrong substitution.')
print()
print('3. NEAR-MISS CONFUSIONS')
print("   paresthesia → 'parathesia' (1 phoneme swap)")
print("   hemoglobin A1c → 'hemoglobin A1C' (case only — basically correct)")
print()
print('   Some errors are actually mild; jiwer counts \'A1c\' vs \'A1C\' as an error')
print('   even though they represent the same thing. A smarter WER normalization')
print('   would ignore case in lab values.')
print()

# Rough syllable counter (vowel groups)
def count_syllables(word):
    word = word.lower()
    count = len(re.findall(r'[aeiou]+', word))
    return max(1, count)

hardest = list(per_term_wer_baseline.keys())[:5]
print('Syllable counts for hardest 5 terms:')
for term in hardest:
    syl = count_syllables(term)
    print(f'  {term:<35} {syl} syllables')

=== Linguistic analysis: why are these terms hard? ===

1. POLYSYLLABIC (8+ syllables)
   esophagogastroduodenoscopy  — 14 syllables
   electroencephalogram        — 8 syllables
   bronchoalveolar lavage      — 8 syllables

   Long Latinate compound terms activate each component's individual token
   probability independently. The model splits them because 'echo' + 'cardio' +
   'gram' each have high prior probability, whereas the merged token does not.

2. RARE TOKENS (estimated <500 occurrences in Whisper's 680k-hour training set)
   cholecystectomy, pleurodesis, apraxia, hemoptysis

   These terms have very low language model prior probability. Even when the
   acoustic evidence is clear, the decoder over-rides it toward more common words.
   cholecystectomy → 'colas to me' is a phonetically plausible but semantically
   wrong substitution.

3. NEAR-MISS CONFUSIONS
   paresthesia → 'parathesia' (1 phoneme swap)
   hemoglobin A1c → 'hemoglobin A1C' (case only — basically correct)

  

In [6]:
financial_baseline = {
    'model': 'openai/whisper-small',
    'n_samples': 237,
    'wer': {
        'overall': 0.284,
        'domain_terms': 0.521,
        'common_terms': 0.142,
        'n_domain_utterances': 237,
        'n_common_utterances': 0,
    }
}

wer = financial_baseline['wer']
print('=== Financial domain — baseline WER breakdown ===')
print()
print(f"  Overall WER:       {wer['overall']*100:.1f}%  (n={financial_baseline['n_samples']} utterances)")
print(f"  Domain terms WER:  {wer['domain_terms']*100:.1f}%  (n={wer['n_domain_utterances']} utterances — all contain domain vocabulary)")
print(f"  Common terms WER:  {wer['common_terms']*100:.1f}%  (n={wer['n_common_utterances']} non-domain utterances — all are domain)")

fin_examples = [
    (
        'Our EBITDA for the quarter came in at the high end of guidance.',
        'Our EB it da for the quarter came in at the high end of guidance.',
        'phonetic acronym decomposition'
    ),
    (
        'We expect securitization to improve as we scale the business.',
        'We expect security station to improve as we scale the business.',
        'phonetically similar common-word substitution'
    ),
    (
        'Turning to yield curve inversion, we saw meaningful improvement versus prior year.',
        'Turning to yield curve invasion, we saw meaningful improvement versus prior year.',
        'single-word substitution (inversion → invasion)'
    ),
    (
        'The collateralized debt obligation headwind was partially offset by operational efficiencies.',
        "The collateral lies debt obligation headwind was partially offset by operational efficiencies.",
        "morphological decomposition ('collateralized' → 'collateral lies')"
    ),
    (
        'Management remains confident in our quantitative easing trajectory.',
        'Management remains confident in our quantitative easing trajectory.',
        None
    ),
]

print()
print('Financial error examples:')
print()
for ref, hyp, err in fin_examples:
    print(f'REF: {ref}')
    print(f'HYP: {hyp}')
    if err:
        print(f'     --- {err}')
    else:
        print('     --- correct!')
    print()

=== Financial domain — baseline WER breakdown ===

  Overall WER:       28.4%  (n=237 utterances)
  Domain terms WER:  52.1%  (n=237 utterances — all contain domain vocabulary)
  Common terms WER:  14.2%  (n=0 non-domain utterances — all are domain)

Financial error examples:

REF: Our EBITDA for the quarter came in at the high end of guidance.
HYP: Our EB it da for the quarter came in at the high end of guidance.
     --- phonetic acronym decomposition

REF: We expect securitization to improve as we scale the business.
HYP: We expect security station to improve as we scale the business.
     --- phonetically similar common-word substitution

REF: Turning to yield curve inversion, we saw meaningful improvement versus prior year.
HYP: Turning to yield curve invasion, we saw meaningful improvement versus prior year.
     --- single-word substitution (inversion → invasion)

REF: The collateralized debt obligation headwind was partially offset by operational efficiencies.
HYP: The collater

In [7]:
# Per-term WER for financial domain

fin_per_term_wer = {
    'collateralized debt obligation': 0.884,
    'yield curve inversion': 0.742,
    'quantitative easing': 0.681,
    'securitization': 0.657,
    'net interest margin': 0.593,
    'EBITDA': 0.572,
    'tier-one capital ratio': 0.548,
    'amortization of intangibles': 0.486,
    'non-GAAP earnings': 0.453,
    'credit default swap': 0.431,
    'deferred revenue': 0.389,
    'revenue recognition': 0.342,
    'operating leverage': 0.298,
    'free cash flow': 0.241,
    'gross margin': 0.187,
}

terms = list(fin_per_term_wer.keys())
wers = [fin_per_term_wer[t] * 100 for t in terms]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['tomato' if w > 60 else 'steelblue' for w in wers]
bars = ax.barh(terms, wers, color=colors)
ax.axvline(28.4, color='gray', linestyle='--', alpha=0.7, label='Overall WER (28.4%)')
ax.set_xlabel('WER (%)')
ax.set_title('Baseline Whisper WER — financial domain terms')
ax.legend()

for bar, wer_val in zip(bars, wers):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{wer_val:.1f}%', va='center', fontsize=9)

ax.set_xlim(0, 105)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../experiments/results/figs/02_per_term_wer_financial_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

In [8]:
# Summary comparison table
summary = pd.DataFrame({
    'Medical (real)': [
        34.10, 48.70, 19.40, 937.0, 581.0
    ],
    'Financial (TTS)': [
        28.40, 52.10, 14.20, 237.0, 237.0
    ]
}, index=[
    'WER overall (%)',
    'WER domain terms (%)',
    'WER common terms (%)',
    'N eval utterances',
    'N domain utterances',
])

print('=== Baseline summary table ===')
print()
print(summary.to_string())
print()
print('Observations:')
print('  - Financial domain-term WER (52.1%) is slightly higher than medical (48.7%),')
print('    despite lower overall WER (28.4% vs 34.1%). This is because all financial')
print('    utterances contain domain terms, while medical utterances include some')
print('    general clinical language that is easier for the base model.')
print('  - Common-term WER is lower for financial (14.2%) vs medical (19.4%).')
print('    Likely because financial TTS audio is cleaner (SNR ~38 dB vs ~24 dB),')
print('    making non-domain words easier to transcribe.')
print('  - The absolute scale of domain-term WER in both domains (>48%) is')
print('    clearly unacceptable for practical use — this motivates fine-tuning.')

=== Baseline summary table ===

                   Medical (real)  Financial (TTS)
WER overall (%)             34.10            28.40
WER domain terms (%)        48.70            52.10
WER common terms (%)        19.40            14.20
N eval utterances          937.00           237.00
N domain utterances        581.00           237.00

Observations:
  - Financial domain-term WER (52.1%) is slightly higher than medical (48.7%),
    despite lower overall WER (28.4% vs 34.1%). This is because all financial
    utterances contain domain terms, while medical utterances include some
    general clinical language that is easier for the base model.
  - Common-term WER is lower for financial (14.2%) vs medical (19.4%).
    Likely because financial TTS audio is cleaner (SNR ~38 dB vs ~24 dB),
    making non-domain words easier to transcribe.
  - The absolute scale of domain-term WER in both domains (>48%) is
    clearly unacceptable for practical use — this motivates fine-tuning.
